<a href="https://colab.research.google.com/github/KerellosRezk231/NewProject/blob/main/ehab%20project%20AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv('ehab.csv')
print(f" تم تحميل الداتا بنجاح. عدد الحالات: {len(df)}")
df.head()

 تم تحميل الداتا بنجاح. عدد الحالات: 1127


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,56.0,male,0.0,125.0,249.0,1.0,0.0,144.0,1.0,1.2,1.0,1.0,2.0,0.0
1,67.0,male,0.0,100.0,299.0,0.0,0.0,125.0,1.0,0.9,1.0,2.0,2.0,0.0
2,52.0,male,0.0,128.0,255.0,0.0,1.0,161.0,1.0,0.0,2.0,1.0,3.0,0.0
3,57.0,male,0.0,152.0,274.0,0.0,1.0,88.0,1.0,0.8,1.0,1.0,3.0,0.0
4,41.0,male,0.0,110.0,172.0,0.0,0.0,158.0,0.0,0.0,2.0,0.0,3.0,0.0


In [5]:
# الشريحة 2: تجهيز البيانات ومعالجة القيم المفقودة (NaN)

# 1. حل مشكلة الخانات الفاضية: بنملى أي مكان فاضي بـ "متوسط" القيم في العمود ده
# السطر ده هو اللي هيشيل الايرور اللي ظهرلك
df.fillna(df.median(numeric_only=True), inplace=True)

# 2. تحويل الجنس لأرقام (ذكر=1، أنثى=0)
if df['sex'].dtype == 'object':
    df['sex'] = df['sex'].map({'male': 1, 'female': 0})

# 3. فصل المدخلات والهدف
X = df.drop('target', axis=1)
y = df['target']

# 4. تقسيم الداتا (80% تدريب - 20% اختبار)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. الميزان (Scaler)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("✅ تم معالجة القيم المفقودة وعمل الـ Scaling بنجاح!")

✅ تم معالجة القيم المفقودة وعمل الـ Scaling بنجاح!


In [6]:
# --- تصحيح معالجة البيانات (السطر ده هو الحل) ---
# بنملى أي خانة فاضية (NaN) بالوسيط (Median) عشان الموديلات تشتغل
df.fillna(df.median(numeric_only=True), inplace=True)

# --- تعريف الـ 3 موديلات والتدريب ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000), # زودنا max_iter عشان يضمن الحل
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
    "SVM": SVC(probability=True, kernel='rbf')
}

print("📊 نتائج المقارنة بعد معالجة البيانات:")
print("-" * 30)

for name, model in models.items():
    # الموديل دلوقتي هيشتغل من غير ValueError لأن الداتا بقت كاملة
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"- دقة {name}: {acc*100:.2f}%")

📊 نتائج المقارنة بعد معالجة البيانات:
------------------------------
- دقة Logistic Regression: 78.32%
- دقة Random Forest: 95.58%
- دقة SVM: 88.50%


In [12]:
# تأكد إنك بتسيف الموديل اللي اخترته والـ scaler اللي استخدمته فعلاً
import joblib

# حفظ الموديل (هنسميه heart_model عشان يمشي مع الـ UI)
joblib.dump(models["Random Forest"], 'heart_model.pkl')

# حفظ الـ scaler اللي احنا دربناه فوق (X_train)
# ملاحظة: استعملنا الـ scaler اللي اتعمله fit فعلياً
joblib.dump(scaler, 'heart_scaler.pkl')

print("✅ تم حفظ الموديل والـ Scaler بنجاح بأسماء متوافقة!")

✅ تم حفظ الموديل والـ Scaler بنجاح بأسماء متوافقة!


In [13]:
%%writefile app_heart.py
import streamlit as st
import joblib
import numpy as np

# تحميل الموديل والميزان
model = joblib.load('heart_model.pkl')
scaler = joblib.load('heart_scaler.pkl')

st.set_page_config(page_title="AI Heart Health", layout="wide")
st.title("❤️ نظام فحص القلب الذكي - إشراف إيهاب")

col1, col2 = st.columns(2)
with col1:
    age = st.number_input("العمر", 1, 120, 50)
    sex = st.selectbox("الجنس", ["ذكر (1)", "أنثى (0)"])
    cp = st.selectbox("نوع ألم الصدر (CP)", [0, 1, 2, 3])
    trestbps = st.number_input("ضغط الدم الانقباضي", 80, 220, 120)
    chol = st.number_input("الكوليسترول", 100, 600, 200)
    fbs = st.selectbox("سكر الدم صائم > 120", [0, 1])

with col2:
    restecg = st.selectbox("رسم القلب", [0, 1, 2])
    thalach = st.number_input("أقصى ضربات قلب", 60, 220, 150)
    exang = st.selectbox("ألم صدر مجهودي؟ (1=نعم)", [0, 1])
    oldpeak = st.number_input("ST depression", 0.0, 6.0, 1.0)
    slope = st.selectbox("ميل قطعة ST", [0, 1, 2])
    ca = st.selectbox("عدد الأوعية الملونة (CA)", [0, 1, 2, 3, 4])
    thal = st.selectbox("حالة الثلاسيميا (Thal)", [0, 1, 2, 3])

if st.button("تحليل الحالة الصحية"):
    sex_val = 1 if "ذكر" in sex else 0
    features = [age, sex_val, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]

    scaled_data = scaler.transform([features])
    prediction = model.predict(scaled_data)
    prob = model.predict_proba(scaled_data)[0][1] * 100

    st.divider()
    if prediction[0] == 1:
        st.error(f"⚠️ احتمالية إصابة مرتفعة ({prob:.1f}%): يرجى استشارة الطبيب.")
    else:
        st.success(f"✅ النتائج مطمئنة: القلب سليم بنسبة ({100-prob:.1f}%).")

Overwriting app_heart.py


In [ ]:
# 1. أول حاجة نطلع الـ IP اللي هنستخدمه كـ Password
import urllib
print("الـ Password المطلوب لفتح الرابط هو:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# 2. تشغيل السيرفر والـ Tunnel مع التأكد من اسم الملف الصح
!streamlit run app_heart.py & npx localtunnel --port 8501

الـ Password المطلوب لفتح الرابط هو: 34.136.92.188
⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://afraid-bees-camp.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.136.92.188:8501

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
